# Lab 5 - ridge, lasso and cross-validation

**Session 5.** Twenty real predictors hidden among eighty noise columns. Use
cross-validation to choose a penalty, then read the lasso's zeros as a feature selection you
have to justify.

## 1. A wide, sparse problem

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

X, y, true_coef = make_regression(
    n_samples=200, n_features=100, n_informative=20, noise=12.0,
    coef=True, random_state=2026,
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=2026)
print(f"train {X_tr.shape}, test {X_te.shape}, "
      f"truly non-zero coefficients: {(true_coef != 0).sum()}")

## 2. Unpenalised least squares, for reference

100 columns and 140 training rows: enough freedom to fit the noise.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


def rmse(model, X, y):
    return float(np.sqrt(mean_squared_error(y, model.predict(X))))


ols = LinearRegression().fit(X_tr, y_tr)
print(f"OLS   train RMSE {rmse(ols, X_tr, y_tr):7.2f}   test RMSE {rmse(ols, X_te, y_te):7.2f}")

## 3. Ridge and lasso, tuned by cross-validation

`RidgeCV`/`LassoCV` run k-fold internally over an alpha grid. Note the scaler: a penalty on
raw coefficients would depend on the units each column happens to use.

In [ ]:
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

alphas = np.logspace(-2, 3, 40)
ridge = Pipeline([("sc", StandardScaler()),
                  ("m", RidgeCV(alphas=alphas, cv=5))]).fit(X_tr, y_tr)
lasso = Pipeline([("sc", StandardScaler()),
                  ("m", LassoCV(alphas=alphas, cv=5, max_iter=20000,
                                random_state=0))]).fit(X_tr, y_tr)

print(f"ridge chose alpha = {ridge['m'].alpha_:.3f}")
print(f"lasso chose alpha = {lasso['m'].alpha_:.3f}")
print(f"ridge train RMSE {rmse(ridge, X_tr, y_tr):7.2f}   test RMSE {rmse(ridge, X_te, y_te):7.2f}")
print(f"lasso train RMSE {rmse(lasso, X_tr, y_tr):7.2f}   test RMSE {rmse(lasso, X_te, y_te):7.2f}")

## 4. The cross-validation curve

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge

means, sds = [], []
for a in alphas:
    scores = -cross_val_score(
        Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=a))]),
        X_tr, y_tr, cv=5, scoring="neg_root_mean_squared_error")
    means.append(scores.mean())
    sds.append(scores.std())
means, sds = np.array(means), np.array(sds)

best = means.argmin()
# One-standard-error rule: the SIMPLEST model (largest alpha) within 1 SE of the best.
threshold = means[best] + sds[best]
one_se = alphas[means <= threshold][-1]

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.errorbar(alphas, means, yerr=sds, marker="o", ms=3, lw=1)
ax.axvline(alphas[best], color="C1", label=f"best alpha = {alphas[best]:.2f}")
ax.axvline(one_se, color="C2", ls="--", label=f"1-SE alpha = {one_se:.2f}")
ax.set(xscale="log", xlabel="alpha (log scale)", ylabel="5-fold CV RMSE",
       title="Ridge: the U-shape, with error bars")
ax.legend()
plt.tight_layout()
plt.show()

## 5. What the lasso selected

Exact zeros are a selection. Check it against the truth - a luxury you only have in a
simulation.

In [ ]:
coef = lasso["m"].coef_
selected = np.flatnonzero(coef)
truly = set(np.flatnonzero(true_coef))
print(f"lasso kept {len(selected)} of 100 columns (truth: {len(truly)})")
print(f"  correctly kept   : {len(set(selected) & truly)}")
print(f"  wrongly kept     : {len(set(selected) - truly)}")
print(f"  wrongly dropped  : {len(truly - set(selected))}")
print(f"\nridge kept all 100 columns; largest |coef| = {np.abs(ridge['m'].coef_).max():.2f}")

## Exercises

1. **The 1-SE rule.** Refit ridge at `one_se` instead of the CV-optimal alpha. How much test
   RMSE does the simpler model cost you? Was it worth it?
2. **Elastic net.** Fit `ElasticNetCV` with `l1_ratio` in `[0.1, 0.5, 0.9, 1.0]`. Where does
   it land, and what does that say about correlated predictors here?
3. **The nested-CV trap.** You just used CV to choose alpha *and* the test set to report.
   Explain in two sentences why reporting the best CV score instead would have been
   optimistic, and what nested CV does about it.